In [1]:
!pip install langchain langchain-community --quiet

from langchain_text_splitters import RecursiveCharacterTextSplitter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.1/476.1 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [14]:
import pandas as pd
from pathlib import Path

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [15]:
import shutil
import os

# Define the path to the repository
repo_path = '/content/ai-research-assistant-rag/'

# Remove the existing cloned directory if it exists
if os.path.exists(repo_path):
    shutil.rmtree(repo_path)
    print(f"Removed existing directory: {repo_path}")
else:
    print(f"Directory {repo_path} does not exist, proceeding with clone.")


Removed existing directory: /content/ai-research-assistant-rag/


In [16]:
# Clone the repository from the 'dev' branch
print("Cloning repository...")
!git clone -b dev https://github.com/harishkulkarni10/ai-research-assistant-rag.git
print("Repository cloned successfully.")


Cloning repository...
Cloning into 'ai-research-assistant-rag'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 51 (delta 9), reused 29 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 18.05 MiB | 20.68 MiB/s, done.
Resolving deltas: 100% (9/9), done.
Repository cloned successfully.


In [17]:
import pandas as pd
from pathlib import Path

# Define the project root and the path to the corpus file
PROJECT_ROOT = Path("/content/ai-research-assistant-rag")
CORPUS_PATH = PROJECT_ROOT / "data" / "processed" / "arxiv_ai_ml_corpus.parquet"

# Load the parquet file into a pandas DataFrame
if CORPUS_PATH.exists():
    print(f"Loading data from: {CORPUS_PATH}")
    df = pd.read_parquet(CORPUS_PATH)
    print(f"Successfully loaded {len(df)} papers.\n")
    print("First 5 rows of the DataFrame:")
    display(df.head())
else:
    print(f"Error: Data file not found at {CORPUS_PATH}. Please check the path and repository structure.")


Loading data from: /content/ai-research-assistant-rag/data/processed/arxiv_ai_ml_corpus.parquet
Successfully loaded 1000 papers.

First 5 rows of the DataFrame:


,paper_id,title,abstract,full_text
0,0,we use new observations of very weak civ absor...,we use new observations of very weak civ absor...,the observational result that high - redshift ...
1,1,we present the detection of a @xmath0 black ho...,we present the detection of a @xmath0 black ho...,the questions of how the nuclei of galaxies fo...
2,2,we study harmonic functions on random environm...,we study harmonic functions on random environm...,"since the work of yau @xcite in 1975 , where t..."
3,3,we study pseudoscalar and scalar mesons using ...,we study pseudoscalar and scalar mesons using ...,dyson - schwinger equations ( dses ) are a non...
4,4,we present our recent results on mm - wave co ...,we present our recent results on mm - wave co ...,"after attending this conference , i think that..."


# Chunking - Let's create token aware chunks
- Our goal is to convert full paper texts into overlapping, semantically correct chunks suitable for dense retrieval
- **Key Design Decisions (Baseline)**
  - Token-based sizing using tiktoken (`cl100k_base`)
  - Chunk size: 1000 tokens
  - Overlap: 200 tokens
  - Min chunk size: 200 tokens (skip tiny fragments)
  - Recursive splitting prioritizing paragraphs → sentences → words
  - Metadata per chunk: `chunk_id`, `paper_id`, `text`, `section` (None for now), `source`, `token_count`, `position_in_paper`

In [18]:
# Chunking parameters
CHUNK_SIZE_TOKENS = 1000        # Target chunk size in tokens
CHUNK_OVERLAP_TOKENS = 200      # Overlap to preserve context across chunks
MIN_CHUNK_SIZE_TOKENS = 200     # Skip very small fragments

In [19]:
!pip install tiktoken --quiet

In [20]:
import tiktoken

In [21]:
tokenizer = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    """Count the number of tokens in a string."""
    return len(tokenizer.encode(text, disallowed_special=()))

sample_text = "The Transformer architecture has revolutionized Natural Language Processing and Language models"
print(f"Token count example: {count_tokens(sample_text)} tokens")
print("Tokenizer ready")

Token count example: 12 tokens
Tokenizer ready


In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size =CHUNK_SIZE_TOKENS,        # 1000 tokens
    chunk_overlap = CHUNK_OVERLAP_TOKENS, # 200 tokens overlap
    length_function = count_tokens,
    separators = ["\n\n", "\n", " ", ""],
    keep_separator=True,
    strip_whitespace=True,
    add_start_index=False
)

print("RecursiveCharacterTextSplitter configured with token-level precision")
print(f"Effective chunk size: ~{CHUNK_SIZE_TOKENS} tokens")
print(f"Overlap: {CHUNK_OVERLAP_TOKENS} tokens")

RecursiveCharacterTextSplitter configured with token-level precision
Effective chunk size: ~1000 tokens
Overlap: 200 tokens


In [23]:
# before scaling to all 1000 papers, let's test the splitter on one paper to verify chunk count,
# chunk quality, confirm token counts are accurate

In [24]:
import numpy as np

sample_row = df.iloc[0]
sample_paper_id = sample_row['paper_id']
sample_text = sample_row['full_text']
sample_title = sample_row['title']

print("Testing on paper ID: {sample_paper_id}")
print(f"Title preview: {sample_title[:200]}...\n")
print(f"Full text token count: {count_tokens(sample_text):,} tokens\n")

# Apply the splitter
chunks = splitter.split_text(sample_text)

print(f"Produced {len(chunks)} chunks")
print(f"Average chunk token count: {np.mean([count_tokens(c) for c in chunks]):.1f}")
print(f"Min / Max chunk tokens: {min([count_tokens(c) for c in chunks])} / {max([count_tokens(c) for c in chunks])}\n")

# Show first chunk
print("=== First Chunk (preview) ===")
print(chunks[0][:800] + "..." if len(chunks[0]) > 800 else chunks[0])

# Show last chunk
print("\n=== Last Chunk (preview) ===")
print(chunks[-1][:800] + "..." if len(chunks[-1]) > 800 else chunks[-1])

# Demonstrate overlap (if >1 chunk)
if len(chunks) > 1:
    print("\n=== Overlap Example ===")
    print("End of first chunk:")
    print(chunks[0][-300:])
    print("\nStart of second chunk:")
    print(chunks[1][:300])

Testing on paper ID: {sample_paper_id}
Title preview: we use new observations of very weak civ absorption lines associated with high - redshift ly @xmath0  absorption systems to measure the high - redshift ly @xmath0  line two - point correlation functio...

Full text token count: 3,705 tokens

Produced 5 chunks
Average chunk token count: 873.4
Min / Max chunk tokens: 595 / 978

=== First Chunk (preview) ===
the observational result that high - redshift ly @xmath0  absorption systems appear not to cluster strongly in redshift ( e.g. sargent 1980 ) has driven most discussion about the origin of the ly @xmath0  forest . 
 this result has generally been interpreted as evidence that high - redshift ly @xmath0  absorbers arise in intergalactic clouds rather than in galaxies . 
 recent studies of the relationship between ly @xmath0  absorbers and galaxies at redshifts @xmath5 , however , directly demonstrate that many or most low - redshift ly @xmath0  absorbers ( or at least those satisfyin

In [25]:
# Let's apply the chunking on all the papers we have i.e 1000

In [26]:
from typing import List, Dict
import tqdm

def create_chunks_for_paper(row: pd.Series) -> List[Dict]:
    """Chunk a single paper and return list of chunk dicts with full metadata"""
    paper_id = row['paper_id']
    text = row['full_text']

    # Split into raw chunks
    raw_chunks = splitter.split_text(text)

    chunks = []
    for idx, chunk_text in enumerate(raw_chunks):
        token_cnt = count_tokens(chunk_text)

        # Skip very small fragments
        if token_cnt < MIN_CHUNK_SIZE_TOKENS:      # if token_cnt < 200
            continue

        chunks.append({
            "chunk_id": f"paper_{paper_id}_chunk_{idx}",
            "paper_id": paper_id,
            "text": chunk_text,
            "section": None,
            "source": "arxiv",
            "token_count": token_cnt,
            "position_in_paper": idx
        })

    return chunks

# Apply to all papers with progress bar
all_chunks = []
for _, row in tqdm.tqdm(df.iterrows(), total=len(df), desc="Chunking papers"):
    all_chunks.extend(create_chunks_for_paper(row))

# Convert to DataFrame
chunks_df = pd.DataFrame(all_chunks)

print(f"\nFull chunking complete!")
print(f"Total chunks created: {len(chunks_df):,}")
print(f"Average chunks per paper: {len(chunks_df) / len(df):.1f}")
print(f"Average token count per chunk: {chunks_df['token_count'].mean():.1f}")
print(f"Total tokens across all chunks: {chunks_df['token_count'].sum():,}")

Chunking papers: 100%|██████████| 1000/1000 [00:36<00:00, 27.74it/s]


Full chunking complete!
Total chunks created: 12,717
Average chunks per paper: 12.7
Average token count per chunk: 903.3
Total tokens across all chunks: 11,486,744


In [27]:
# Let's validate the chunking we did

In [33]:
# Chunk distribution by paper
chunks_per_paper = chunks_df.groupby('paper_id').size()
print(f"Chunks per paper — Min: {chunks_per_paper.min()}, Max: {chunks_per_paper.max()}, Mean: {chunks_per_paper.mean():.1f}")
print(f"Papers with >20 chunks: {(chunks_per_paper > 20).sum()}")
print(f"Papers with <5 chunks: {(chunks_per_paper < 5).sum()}\n")

# Token stats
print("Token count stats:")
print(chunks_df['token_count'].describe()[['mean', 'std', 'min', '25%', '50%', '75%', 'max']])

# Let's sample a few chunks for manual inspection
print("\n=== Sample Chunk 1 ===")
sample1 = chunks_df.sample(1, random_state=42).iloc[0]
print(f"Chunk ID: {sample1['chunk_id']}")
print(f"Paper ID: {sample1['paper_id']}")
print(f"Token count: {sample1['token_count']}")
print(f"Position: {sample1['position_in_paper']}")
print("\nText preview:")
print(sample1['text'][:1000] + "..." if len(sample1['text']) > 1000 else sample1['text'])

print("\n=== Sample Chunk 2 ===")
sample2 = chunks_df.sample(1, random_state=123).iloc[0]
print(f"Chunk ID: {sample2['chunk_id']}")
print(f"Token count: {sample2['token_count']}")
print("\nText preview:")
print(sample2['text'][:1000] + "..." if len(sample2['text']) > 1000 else sample2['text'])

Chunks per paper — Min: 1, Max: 336, Mean: 12.8
Papers with >20 chunks: 136
Papers with <5 chunks: 122

Token count stats:
mean     903.258945
std      134.828967
min      200.000000
25%      919.000000
50%      949.000000
75%      964.000000
max     1000.000000
Name: token_count, dtype: float64

=== Sample Chunk 1 ===
Chunk ID: paper_506_chunk_3
Paper ID: 506
Token count: 967
Position: 3

Text preview:
we assume that the spatial distribution of the cluster plasma is described by a spherical isothermal @xmath6-model ( cavaliere and fusco - femiano 1976 , 1978 ) given by : @xmath21 where @xmath2 is the electron number density , @xmath22 is the radius from the cluster s center , @xmath23 is the core radius and @xmath6 is a power - law index . with this model , 
 the thermodynamic sze temperature decrement / increment @xmath24 is    @xmath25    where @xmath26 is the frequency dependence of the sze with @xmath27 ( e.g. , reese et al . 
 2002 and references therein ) , @xmath28 is the clust

In [34]:
# Save to the repo (processed folder)
chunks_output_path = PROJECT_ROOT / "data" / "processed" / "chunks.parquet"

chunks_df.to_parquet(chunks_output_path, index=False)

print(f"Chunks artifact saved!")
print(f"Location: {chunks_output_path}")
print(f"File size: {chunks_output_path.stat().st_size / 1e6:.1f} MB")

# VERIFY
verify_df = pd.read_parquet(chunks_output_path)
print(f"Verification: {len(verify_df):,} chunks loaded back successfully")

Chunks artifact saved!
Location: /content/ai-research-assistant-rag/data/processed/chunks.parquet
File size: 18.9 MB
Verification: 12,717 chunks loaded back successfully


In [38]:
!cd /content/ai-research-assistant-rag && git remote set-url origin https://<YOUR_GITHUB_TOKEN>@github.com/harishkulkarni10/ai-research-assistant-rag.git

!cd /content/ai-research-assistant-rag && git push origin dev

Enumerating objects: 8, done.
Counting objects: 100% (8/8), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (5/5), 15.79 MiB | 7.06 MiB/s, done.
Total 5 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/harishkulkarni10/ai-research-assistant-rag.git
   fb075f1..aaff687  dev -> dev
